# DrugBank SMILES Extraction

This notebook extracts DrugBank IDs and SMILES strings from the DrugBank full database XML file.

**Input:** `full database.xml` (from DrugBank 'All Drugs' download)  
**Output:** `drugbank_smiles.csv` with columns `drugbank_id`, `smiles`

Run this notebook first before running the KNN baseline notebook.

## Imports

In [3]:
import xml.etree.ElementTree as ET
import pandas as pd

print('Imports successful')

Imports successful


## Step 1 — Parse DrugBank XML

Make sure `full database.xml` is in the same folder as this notebook before running.

This may take 1–2 minutes as the file is large.

In [4]:
XML_FILE = 'drugbank_full_database.xml'  # update this if your filename is different

print(f'Parsing {XML_FILE}... (this may take a minute)')

tree = ET.parse(XML_FILE)
root = tree.getroot()

print('Parsing complete.')

Parsing drugbank_full_database.xml... (this may take a minute)
Parsing complete.


## Step 2 — Extract DrugBank IDs and SMILES

In [5]:
# DrugBank XML uses a namespace prefix on all tags
ns = '{http://www.drugbank.ca}'

rows = []
missing_smiles = 0

for drug in root.findall(f'{ns}drug'):
    drugbank_id = None
    smiles = None

    # Get primary DrugBank ID (e.g. DB00001)
    for did in drug.findall(f'{ns}drugbank-id'):
        if did.get('primary') == 'true':
            drugbank_id = did.text
            break

    # Get SMILES from calculated-properties section
    calc_props = drug.find(f'{ns}calculated-properties')
    if calc_props:
        for prop in calc_props.findall(f'{ns}property'):
            kind  = prop.find(f'{ns}kind')
            value = prop.find(f'{ns}value')
            if kind is not None and kind.text == 'SMILES':
                smiles = value.text
                break

    if drugbank_id and smiles:
        rows.append({'drugbank_id': drugbank_id, 'smiles': smiles})
    elif drugbank_id:
        missing_smiles += 1

df = pd.DataFrame(rows)

print(f'Drugs with SMILES:         {len(df):,}')
print(f'Drugs missing SMILES:      {missing_smiles:,}')
print(f'\nSample output:')
df.head()

/var/folders/27/dgsnrqdd711fyh394v18jvnw0000gn/T/ipykernel_37312/589562861.py:19: DeprecationWarning: Testing an element's truth value will always return True in future versions.  Use specific 'len(elem)' or 'elem is not None' test instead.
  if calc_props:


Drugs with SMILES:         14,627
Drugs missing SMILES:      5,230

Sample output:


,drugbank_id,smiles
0,DB00006,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...
1,DB00014,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...
2,DB00027,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...
3,DB00035,NC(=O)CC[C@@H]1NC(=O)[C@H](CC2=CC=CC=C2)NC(=O)...
4,DB00050,CC(C)C[C@H](NC(=O)[C@@H](CCCNC(N)=O)NC(=O)[C@H...


## Step 3 — Save to CSV

In [6]:
df.to_csv('drugbank_smiles.csv', index=False)
print(f'Saved {len(df):,} rows to drugbank_smiles.csv')
print('\nYou can now run the ChemBERTa embedding notebook.')

Saved 14,627 rows to drugbank_smiles.csv

You can now run the ChemBERTa embedding notebook.
